In [1]:
import cobra
import pandas
from cobra.io import read_sbml_model, write_sbml_model
import logging
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis import gapfill
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from cobra import Model, Reaction, Metabolite
from copy import deepcopy
from collections import defaultdict
from cobra.io import load_json_model

In [2]:
#Function based on https://github.com/opencobra/cobrapy/issues/707 and completely altered by me!

def removeDuplicateRxn(model):
    model2 = deepcopy(model)
    toRemove = []
    doubt = []
    
    for eachReaction in model.reactions:

        if(eachReaction.id not in toRemove):

            ids = []
            stechiometry = []
            gn = []
        
            #Placing metabolites and genes of R1 in list
            for eachMet in eachReaction.metabolites:
                ids.append(eachMet.id)
                stechiometry.append(eachReaction.metabolites[eachMet])

            ids.sort()
        
            for eachGene in eachReaction.genes:
                gn.append(eachGene.id)
    
        #Starting comparison
            for eachReaction2 in model2.reactions:
            
                if(eachReaction2.id not in toRemove):
                           
                    if eachReaction.id != eachReaction2.id: #Comparing ids to avoid self comparison
                
                        ids2=[]
                        stechiometry2 = []

                        for eachMet2 in eachReaction2.metabolites:
                            ids2.append(eachMet2.id)
                            stechiometry2.append(eachReaction2.metabolites[eachMet2])
                
                        ids2.sort()    
                
                        if(ids == ids2): #all metabolites are the same
                    
                            #Comparing genes
                            duplicate = 0
                            unsure = 0
                        
                            if(len(gn) == 0 or len(eachReaction2.genes) == 0): #one of the reactions don't have associated genes
                                unsure = 1

                            else:
                                for eachGene2 in eachReaction2.genes:
                                    if(eachGene2.id in gn): #At least one gene is the same
                                        duplicate = 1
                                        break
                        
                            if(duplicate == 1): #Checking if any reaction is reversible and removing the non-reversible one
                                if(eachReaction.lower_bound != 0 and eachReaction.upper_bound != 0): 
                                    toRemove.append(eachReaction2.id) 
                                elif(eachReaction2.lower_bound != 0 and eachReaction2.upper_bound != 0):
                                    toRemove.append(eachReaction.id)
                                else:
                                    unsure = 1
                                    
                            if(unsure == 1):
                                td=[]
                                td.append(eachReaction.id)
                                td.append(eachReaction2.id)
                                td.sort()
                                doubt.append(str(td[0]+":"+td[1]))
                                
                
    a = list(set(doubt))
    model2.remove_reactions(toRemove)
    model2.repair()
    return[model2,toRemove,a]

In [3]:
#Changing configuration
cobra_config = cobra.Configuration()
cobra_config.bounds = -999999.0,999999.0
cobra_config.solver = "cplex"

In [4]:
#Loading BiGG's universal model for gapfilling

universal = load_json_model("/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/carveme/data/generated/universal_model_cobrapy.json")

In [5]:
#Setting inputfile
input = "/scr/k61san/natasha/matomic/trials/CarveMe/Eramosum.tcds.top3.gramPosN.cim8.xml"

In [6]:
#Read carveme model
model = cobra.io.read_sbml_model(str(input))

In [7]:
#Fixing masses
model.metabolites.get_by_id("23dhb_c").formula = "C7H6O4"
model.metabolites.get_by_id("ACP_c").formula = "C11H21N2O7PRS"
model.metabolites.get_by_id("benzcoa_c").formula = "C28H36N7O17P3S"
model.metabolites.get_by_id("d23hb_e").formula = "C7H7O4"
model.metabolites.get_by_id("fad_c").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fmn_c").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmnh2_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("Nforglu_c").formula = "C6H7NO5"
model.metabolites.get_by_id("palmACP_c").formula = "C27H51N2O8PRS"
model.metabolites.get_by_id("ribflv_c").formula = "C17H20N4O6"
model.metabolites.get_by_id("ribflv_e").formula = "C17H20N4O6"

In [8]:
#Fixing charges
model.metabolites.get_by_id("23ddhb_c").charge = -1
model.metabolites.get_by_id("23dhb_c").charge = 0
model.metabolites.get_by_id("2agpg120_c").charge = -1
model.metabolites.get_by_id("2agpg120_p").charge = -1
model.metabolites.get_by_id("2agpg180_c").charge = -1
model.metabolites.get_by_id("2agpg180_p").charge = -1
model.metabolites.get_by_id("2ahbut_c").charge = -1
model.metabolites.get_by_id("2dr1p_c").charge = -2
model.metabolites.get_by_id("2me4p_c").charge = -2
model.metabolites.get_by_id("2shchc_c").charge = -2
model.metabolites.get_by_id("5aizc_c").charge = -3
model.metabolites.get_by_id("6pgg_c").charge = -2
model.metabolites.get_by_id("acgam1p_c").charge = -2
model.metabolites.get_by_id("acmanap_c").charge = -2
model.metabolites.get_by_id("acmum6p_c").charge = -3
model.metabolites.get_by_id("ACP_c").charge = -1
model.metabolites.get_by_id("air_c").charge = -2
model.metabolites.get_by_id("ametam_c").charge = 2
model.metabolites.get_by_id("argsuc_c").charge = -1
model.metabolites.get_by_id("dcamp_c").charge = -4
model.metabolites.get_by_id("dhpmp_c").charge = -2
model.metabolites.get_by_id("dscl_c").charge = -7
model.metabolites.get_by_id("fad_c").charge = -2
model.metabolites.get_by_id("fdp_c").charge = -4
model.metabolites.get_by_id("fdxrd_c").charge = 0
model.metabolites.get_by_id("fgam_c").charge = -2
model.metabolites.get_by_id("fmn_c").charge = -2
model.metabolites.get_by_id("fmnh2_c").charge = -2
model.metabolites.get_by_id("fpram_c").charge = -2 #nao mudar
model.metabolites.get_by_id("fruur_c").charge = -1
model.metabolites.get_by_id("g3p_c").charge = -2
model.metabolites.get_by_id("g3pg_c").charge = -1
model.metabolites.get_by_id("g3pg_e").charge = -1
model.metabolites.get_by_id("g3pg_p").charge = -1
model.metabolites.get_by_id("gdptp_c").charge = -7
model.metabolites.get_by_id("man1p_c").charge = -2
model.metabolites.get_by_id("man6p_c").charge = -2
model.metabolites.get_by_id("man6pglyc_c").charge = -3
model.metabolites.get_by_id("Nforglu_c").charge = -2
model.metabolites.get_by_id("pa120_c").charge = -2
model.metabolites.get_by_id("pa140_c").charge = -2
model.metabolites.get_by_id("pa161_c").charge = -2
model.metabolites.get_by_id("pa180_c").charge = -2
model.metabolites.get_by_id("pa181_c").charge = -2
model.metabolites.get_by_id("Pald_c").charge = -2
model.metabolites.get_by_id("palmACP_c").charge = -1
model.metabolites.get_by_id("peptido_BS_c").charge = -2
model.metabolites.get_by_id("pg120_c").charge = -1
model.metabolites.get_by_id("pg120_p").charge = -1
model.metabolites.get_by_id("pg160_c").charge = -1
model.metabolites.get_by_id("pg161_c").charge = -1
model.metabolites.get_by_id("pg161_p").charge = -1
model.metabolites.get_by_id("pg180_c").charge = -1
model.metabolites.get_by_id("pg180_p").charge = -1
model.metabolites.get_by_id("pphn_c").charge = -2
model.metabolites.get_by_id("pppi_c").charge = -4
model.metabolites.get_by_id("pqq_p").charge = -3
model.metabolites.get_by_id("pqqh2_p").charge = -3
model.metabolites.get_by_id("prbamp_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("ps160_c").charge = -1
model.metabolites.get_by_id("r5p_c").charge = -2
model.metabolites.get_by_id("sbt6p_c").charge = -2
model.metabolites.get_by_id("sbzcoa_c").charge = -5
model.metabolites.get_by_id("scl_c").charge = -7
model.metabolites.get_by_id("suc6p_c").charge = -2
model.metabolites.get_by_id("tag6p__D_c").charge = -2
model.metabolites.get_by_id("tagdp__D_c").charge = -4
model.metabolites.get_by_id("tdecoa_c").charge = -4
model.metabolites.get_by_id("trnaglu_c").charge = 0
model.metabolites.get_by_id("tsul_c").charge = -2
model.metabolites.get_by_id("tsul_p").charge = -2
model.metabolites.get_by_id("uaagmda_c").charge = -4
model.metabolites.get_by_id("uagmda_c").charge = -4
model.metabolites.get_by_id("uamr_c").charge = -3
model.metabolites.get_by_id("udcpdp_c").charge = -3
model.metabolites.get_by_id("udcpdp_e").charge = -3
model.metabolites.get_by_id("udcpp_c").charge = -2
model.metabolites.get_by_id("udcpp_e").charge = -2
model.metabolites.get_by_id("udpacgal_c").charge = -2

In [9]:
# Identifying and removing duplicated reactions
# md model with removed reactions
# rd removed list
# dt reactions in doubt

[md,rd, dt] = removeDuplicateRxn(model)

In [10]:
#Removing reactions in doubt after manual inspection
md.remove_reactions([md.reactions.get_by_id("EX_abt__L_e"),md.reactions.get_by_id("EX_isetac_e"),
                     md.reactions.get_by_id("EX_glcn__D_e"),md.reactions.get_by_id("EX_ethso3_e"),
                     md.reactions.get_by_id("EX_galctr__D_e"),md.reactions.get_by_id("EX_sulfac_e"),
                     md.reactions.get_by_id("EX_orn__L_e"),md.reactions.get_by_id("EX_metsox_S__L_e"),
                     md.reactions.get_by_id("PSUDS"),md.reactions.get_by_id("MS_1"),
                     md.reactions.get_by_id("URIK2_1"),md.reactions.get_by_id("PRFGCL"),
                     md.reactions.get_by_id("URIK3_1"),md.reactions.get_by_id("PRAIS"),
                     md.reactions.get_by_id("TRPAS1"),md.reactions.get_by_id("ASPCT_2"),
                     md.reactions.get_by_id("CBPS_1"),md.reactions.get_by_id("CYTDK2_1"),
                     md.reactions.get_by_id("CYTDK1_1"),md.reactions.get_by_id("NP1"),
                     md.reactions.get_by_id("NADK_1"),md.reactions.get_by_id("UAGDP_1"),
                     md.reactions.get_by_id("URIK1_1")])

In [11]:
#Running FVA to id blocked reactions
#Universally blocked reactions are reactions that during Flux Variability Analysis cannot carry any flux while all 
#model boundaries are open. Generally blocked reactions are caused by network gaps, which can be attributed to 
#scope or knowledge gaps. 
flux_variability_analysis(md,loopless=True)

/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


,minimum,maximum
24DECOAR,0.000000,0.000000e+00
26DAHtex,0.000000,0.000000e+00
2AGPE120tipp,0.000000,1.158851e-12
2AGPE141tipp,0.000000,1.136993e-12
2AGPE160tipp,0.000000,0.000000e+00
...,...,...
DHORD6,0.051218,5.121843e-02
SUCBZL_1,0.000015,1.548343e-05
SUCBZS,0.000015,1.548343e-05
SULR_1,3.388322,6.613755e+02


In [12]:
#Performing gapfill with universal model from BiGG to try to reduce blocked reactions
#Usually, it does not do anything (and it takes some time to run)

gapfill(md, universal, exchange_reactions=True, demand_reactions=False, iterations=10)

[[], [], [], [], [], [], [], [], [], []]

In [13]:
#Adding/Removing/Editing reactions manually to reduce blocked reactions

md.remove_reactions([md.reactions.get_by_id("CAt6pp"),md.reactions.get_by_id("MG2tex"),
                     md.reactions.get_by_id("EX_eths_e"),md.reactions.get_by_id("EX_galct__D_e"),
                    md.reactions.get_by_id("EX_glcn_e"),md.reactions.get_by_id("EX_istnt_e"),
                    md.reactions.get_by_id("EX_sula_e")]) #nothing is done with ca or mg in periplasm
md.remove_metabolites([md.metabolites.get_by_id("ca2_p"),md.metabolites.get_by_id("mg2_p"),
                       md.metabolites.get_by_id("eths_e"),md.metabolites.get_by_id("galct__D_e"),
                      md.metabolites.get_by_id("glcn_e"),md.metabolites.get_by_id("istnt_e"),
                      md.metabolites.get_by_id("sula_e"),md.metabolites.get_by_id("12ppd__S_c"),
                       md.metabolites.get_by_id("lac__L_c"),md.metabolites.get_by_id("lald__L_c"),
                       md.metabolites.get_by_id("na1_p")]) #nothing is done with sulfur

md.remove_reactions([md.reactions.get_by_id("EX_met__D_e"),md.reactions.get_by_id("METte"),
                     md.reactions.get_by_id("METDabc")]) #nothing is done with sulfur
md.remove_metabolites([md.metabolites.get_by_id("met__D_e"),md.metabolites.get_by_id("s_c")]) #nothing is done with sulfur

md.remove_reactions([md.reactions.get_by_id("St"),md.reactions.get_by_id("EX_s_e")]) #nothing is done with sulfur
md.remove_metabolites([md.metabolites.get_by_id("s_e"),md.metabolites.get_by_id("met__D_c")]) #nothing is done with sulfur

md.remove_reactions([md.reactions.get_by_id("EX_chol_e"),md.reactions.get_by_id("CHLabc"),md.reactions.get_by_id("CHLabc_rev")]) #nothing is done with choline
md.remove_metabolites([md.metabolites.get_by_id("chol_e"),md.metabolites.get_by_id("chol_c")]) #nothing is done with choline

md.add_reactions([universal.reactions.get_by_id("GALabc"),universal.reactions.get_by_id("ABTt")]) #Adding missing transport to cytoplasm

md.remove_reactions([md.reactions.get_by_id("LCARS")]) #Not connected to the network

md.remove_reactions([md.reactions.get_by_id("LDH_L")]) #nothing is done with L-lactate

md.remove_reactions([md.reactions.get_by_id("YUMPS")])
md.remove_metabolites([md.metabolites.get_by_id("psd5p_c")])

md.remove_reactions([md.reactions.get_by_id("TRPTA")])
md.remove_metabolites([md.metabolites.get_by_id("indpyr_c")])

md.remove_reactions([md.reactions.get_by_id("PPAKr")])
md.remove_metabolites([md.metabolites.get_by_id("ppa_c")])

md.remove_reactions([md.reactions.get_by_id("NP1_1")])
md.remove_metabolites([md.metabolites.get_by_id("nicrns_c")])

md.remove_reactions([md.reactions.get_by_id("MHPGLUT")])
md.remove_metabolites([md.metabolites.get_by_id("mhpglu_c"),md.metabolites.get_by_id("hpglu_c")])

md.repair()

In [14]:
#Running FVA again after gapfilling
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arg__L_e,EX_arg__L_e,0.0458,[0; 0.04588],6,0.33%
asn__L_e,EX_asn__L_e,0.1997,[0; 0.4842],4,0.95%
ca2_e,EX_ca2_e,0.0008059,[0.0007656; 0.0008059],0,0.00%
cl_e,EX_cl_e,0.0008059,[0.0007656; 0.0008059],0,0.00%
cobalt2_e,EX_cobalt2_e,1.548E-05,[1.471E-05; 1.542E-05],0,0.00%
cu2_e,EX_cu2_e,0.0001098,[0.0001043; 0.0001098],0,0.00%
fe3_e,EX_fe3_e,10,[9.5; 10],0,0.00%
fol_e,EX_fol_e,0.0001036,[9.84E-05; 0.0001036],19,0.00%
fru_e,EX_fru_e,10,[2.761; 10],6,71.61%
glc__D_e,EX_glc__D_e,3.546,[2.761; 10],6,25.39%


In [15]:
rlist = [md.reactions.get_by_id("CO2t"),md.reactions.get_by_id("CO2tex"),md.reactions.get_by_id("CO2tpp")]
flux_variability_analysis(md,rlist,loopless=True)

,minimum,maximum
CO2t,-26.111548,0.0
CO2tex,-26.111548,0.0
CO2tpp,-26.111548,0.0


In [16]:
output = input.replace("xml", "")
output

'/scr/k61san/natasha/matomic/trials/CarveMe/Eramosum.tcds.top3.gramPosN.cim8.'

In [17]:
#Writting intermediate network
sbml = output + "manual.xml"
print(sbml)
cobra.io.write_sbml_model(md, sbml)

/scr/k61san/natasha/matomic/trials/CarveMe/Eramosum.tcds.top3.gramPosN.cim8.manual.xml
